# Natural-Language Data Preparation: Observation and Approval Tree

This notebook executes the read-only data checks, then pauses at each choice that could affect cleaning or features. You approve or disapprove each proposal in its own code cell. A disapproval pauses that branch; no alternative cleaning action is guessed.

## Natural-language request

> Load all bookings, split them using the project arrival-date periods, inspect the data, and show each proposed cleaning or feature action with its evidence. Wait for my approval or disapproval before treating any proposal as final. Do not inspect proof cancellation outcomes or train a model.

**Already requested and executed:** split by the contract periods. **Still awaiting your approval:** handling exact repeats, missing values, unusual values, candidate feature timing, log transforms, and hotel encoding.

Edit each `decision_...` cell to `"approve"` or `"disapprove"`, then run that cell and the final approval summary. `"pending"` makes no choice.

## 1. Load the raw file and supplied contract helpers

The raw table is kept unchanged. We add the arrival date to a separate working copy.

In [1]:
from pathlib import Path
import sys

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import numpy as np
import pandas as pd

raw_bookings = data.read_raw()
bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)

print(f"Rows: {len(raw_bookings):,}")
print(f"Source columns: {raw_bookings.shape[1]}")
print(f"Arrival dates: {bookings['arrival_date'].min().date()} to {bookings['arrival_date'].max().date()}")

Rows: 119,390
Source columns: 32
Arrival dates: 2015-07-01 to 2017-08-31


## 2. Split by arrival date

The contract function sets the boundaries. The summary below does not calculate cancellation rates.

In [2]:
train, choose, proof = contract.split_periods(bookings)

def summarize_period(name, period):
    """Return rows, dates, and hotel names without inspecting outcomes."""
    return {
        "period": name,
        "rows": len(period),
        "first_arrival": period["arrival_date"].min().date(),
        "last_arrival": period["arrival_date"].max().date(),
        "hotels": sorted(period["hotel"].dropna().unique()),
    }

period_summary = pd.DataFrame([
    summarize_period("train", train),
    summarize_period("choose", choose),
    summarize_period("proof", proof),
])
print(period_summary.to_string(index=False))

print(f"Rows assigned: {len(train) + len(choose) + len(proof):,} of {len(bookings):,}")

period  rows first_arrival last_arrival                     hotels
 train 49157    2015-07-01   2016-06-30 [City Hotel, Resort Hotel]
choose 29546    2016-07-01   2016-12-31 [City Hotel, Resort Hotel]
 proof 40687    2017-01-01   2017-08-31 [City Hotel, Resort Hotel]
Rows assigned: 119,390 of 119,390


## 3. Natural-language request: profile training data quality

> On the training period only, show exact duplicate rows, columns with missing values, and simple range checks. Do not remove rows or fill values yet.

This keeps proof outcomes out of the EDA. Duplicate rows are not automatically errors because the file has no reservation ID to distinguish a repeated record from two identical bookings.

In [3]:
print(f"Training shape: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"Exact repeated rows after first occurrence: {train.duplicated().sum():,}")

missing_counts = train.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("Missing values in training data:")
print(missing_counts.to_string())

range_columns = [
    "adults", "children", "babies",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adr", "previous_cancellations", "previous_bookings_not_canceled",
]
range_summary = train[range_columns].agg(["min", "max"]).T
range_summary["negative_values"] = train[range_columns].lt(0).sum()
print("Range checks (training period):")
print(range_summary.to_string())

Training shape: 49,157 rows x 33 columns
Exact repeated rows after first occurrence: 16,291
Missing values in training data:
company     46123
agent        6960
country       284
children        4
Range checks (training period):
                                min     max  negative_values
adults                          0.0    55.0                0
children                        0.0    10.0                0
babies                          0.0    10.0                0
stays_in_weekend_nights         0.0    18.0                0
stays_in_week_nights            0.0    42.0                0
adr                             0.0  5400.0                0
previous_cancellations          0.0    26.0                0
previous_bookings_not_canceled  0.0    36.0                0


### 3a. Profile all source columns on train

This compact profile shows type, missingness, distinct values, and the three most common values for every source column. The numeric table adds min, median, 99th percentile, max, and negative-value count. All statistics here use train only.

In [4]:
train_source = train[raw_bookings.columns]
column_rows = []
for column in train_source.columns:
    values = train_source[column]
    common_values = values.value_counts(dropna=False).head(3).to_dict()
    column_rows.append({
        "column": column,
        "type": str(values.dtype),
        "missing": int(values.isna().sum()),
        "distinct": int(values.nunique(dropna=True)),
        "top_values": str(common_values),
    })

column_profile = pd.DataFrame(column_rows)
print(column_profile.to_string(index=False))

numeric_columns = train_source.select_dtypes(include="number").columns
numeric_profile = train_source[numeric_columns].quantile([0, 0.5, 0.99, 1]).T
numeric_profile.columns = ["min", "median", "p99", "max"]
numeric_profile["negative_values"] = train_source[numeric_columns].lt(0).sum()
print("\nNumeric source-column summary:")
print(numeric_profile.to_string())

                        column    type  missing  distinct                                                       top_values
                         hotel     str        0         2                     {'City Hotel': 31623, 'Resort Hotel': 17534}
                   is_canceled   int64        0         2                                             {0: 31572, 1: 17585}
                     lead_time   int64        0       379                                      {0: 3338, 1: 1764, 2: 1021}
             arrival_date_year   int64        0         2                                       {2016: 27161, 2015: 21996}
            arrival_date_month     str        0        12                       {'May': 5478, 'April': 5428, 'June': 5292}
      arrival_date_week_number   int64        0        53                                   {25: 1468, 21: 1452, 18: 1437}
     arrival_date_day_of_month   int64        0        31                                    {17: 2147, 5: 1975, 19: 1852}
       stays_in_

### 3b. Check cancellation patterns by training month

This checks whether the target varies over the training timeline and by hotel. It does not use choose or proof outcomes.

In [5]:
train_monthly = (
    train.groupby([train["arrival_date"].dt.to_period("M"), "hotel"])["is_canceled"]
    .agg(["count", "mean"])
    .rename(columns={"count": "bookings", "mean": "cancellation_rate"})
)
print(train_monthly.to_string(formatters={"cancellation_rate": "{:.1%}".format}))

                           bookings cancellation_rate
arrival_date hotel                                   
2015-07      City Hotel        1398             67.2%
             Resort Hotel      1378             23.2%
2015-08      City Hotel        2480             49.7%
             Resort Hotel      1409             26.0%
2015-09      City Hotel        3529             43.7%
             Resort Hotel      1585             34.8%
2015-10      City Hotel        3386             39.0%
             Resort Hotel      1571             26.2%
2015-11      City Hotel        1235             24.4%
             Resort Hotel      1105             16.7%
2015-12      City Hotel        1654             40.4%
             Resort Hotel      1266             24.1%
2016-01      City Hotel        1364             32.1%
             Resort Hotel       884             13.5%
2016-02      City Hotel        2371             39.2%
             Resort Hotel      1520             26.8%
2016-03      City Hotel     

## 4. Observation: inspect training target and candidate fields

The target and monthly/hotel pattern below use train only. Candidate features are provisional because a Bin 1 judgment is not independent proof of when the source system makes a value available.

In [6]:
target_counts = train["is_canceled"].value_counts().sort_index()
print("Training target counts:")
print(target_counts.to_string())
print(f"Training cancellation rate: {train['is_canceled'].mean():.1%}")

hotel_target_summary = train.groupby("hotel")["is_canceled"].agg(["count", "mean"])
hotel_target_summary = hotel_target_summary.rename(columns={"count": "bookings", "mean": "cancellation_rate"})
print("Training target by hotel:")
print(hotel_target_summary.to_string(formatters={"cancellation_rate": "{:.1%}".format}))

candidate_columns = [
    "hotel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
]
candidate_profile = train[candidate_columns].agg(["count"]).T
candidate_profile["missing"] = train[candidate_columns].isna().sum()
candidate_profile["distinct_values"] = train[candidate_columns].nunique(dropna=True)
candidate_profile["zero_rows"] = train[candidate_columns].eq(0).sum()
print("Provisional candidate fields:")
print(candidate_profile.to_string())

Training target counts:
is_canceled
0    31572
1    17585
Training cancellation rate: 35.8%
Training target by hotel:
              bookings cancellation_rate
hotel                                   
City Hotel       31623             41.7%
Resort Hotel     17534             25.2%
Provisional candidate fields:
                                count  missing  distinct_values  zero_rows
hotel                           49157        0                2          0
is_repeated_guest               49157        0                2      47529
previous_cancellations          49157        0               14      43348
previous_bookings_not_canceled  49157        0               37      47813


## 5. Proposed data-handling decisions - awaiting your approval

The notebook has not removed rows, filled values, or clipped numbers. These are proposals based on the observations, not approved cleaning actions. The detailed branches are below.

### Evidence summary and proposed data actions

Review this table, then answer each decision gate below.

In [7]:
candidate_missing = pd.DataFrame({
    "train": train[candidate_columns].isna().sum(),
    "choose": choose[candidate_columns].isna().sum(),
    "proof": proof[candidate_columns].isna().sum(),
})
print("Missing values in provisional inputs by split:")
print(candidate_missing.to_string())

proposed_data_actions = pd.DataFrame([
    {"proposal": "Retain exact repeated rows", "evidence": "16,291 exact repeated train rows; no booking ID to confirm errors", "status": "Awaiting approval"},
    {"proposal": "Leave missing non-candidate fields unchanged", "evidence": "company, agent, country, children have missing values; candidate fields do not", "status": "Awaiting approval"},
    {"proposal": "Do not clip unusual non-negative values", "evidence": "No negatives in checked train fields; large values alone do not prove errors", "status": "Awaiting approval"},
])
print("\nProposals only; no data values changed:")
print(proposed_data_actions.to_string(index=False))

Missing values in provisional inputs by split:
                                train  choose  proof
hotel                               0       0      0
is_repeated_guest                   0       0      0
previous_cancellations              0       0      0
previous_bookings_not_canceled      0       0      0

Proposals only; no data values changed:
                                    proposal                                                                       evidence            status
                  Retain exact repeated rows              16,291 exact repeated train rows; no booking ID to confirm errors Awaiting approval
Leave missing non-candidate fields unchanged company, agent, country, children have missing values; candidate fields do not Awaiting approval
     Do not clip unusual non-negative values   No negatives in checked train fields; large values alone do not prove errors Awaiting approval


## 6. Proposed feature set and engineering - awaiting your approval

> Candidate pool: the four fields you classified as Bin 1. Proposed preparation: one-hot encode `hotel` from train categories and add `log1p` versions of the prior booking counts. Do not include `is_canceled`, `reservation_status`, or `reservation_status_date` as predictors.

The feature matrices below are preview only. They are not final selected inputs and must not be passed to a model until you approve the relevant gates. Point-in-time availability still needs verification.

In [8]:
hotel_categories = sorted(train["hotel"].dropna().unique())

def make_features(period):
    """Build the same candidate features without reading the target."""
    features = period[candidate_columns].copy()
    features["log_previous_cancellations"] = np.log1p(features["previous_cancellations"])
    features["log_previous_bookings_not_canceled"] = np.log1p(features["previous_bookings_not_canceled"])
    features = features.drop(columns="hotel")

    for hotel_name in hotel_categories:
        column_name = "hotel_" + hotel_name.lower().replace(" ", "_")
        features[column_name] = period["hotel"].eq(hotel_name).astype(int)

    return features

# Prepare features separately for each time period. No target is passed in.
X_train_preview = make_features(train)
X_choose_preview = make_features(choose).reindex(columns=X_train_preview.columns, fill_value=0)
X_proof_preview = make_features(proof).reindex(columns=X_train_preview.columns, fill_value=0)

print("Feature columns:")
print(X_train_preview.columns.tolist())
print(f"Train feature shape: {X_train_preview.shape}")
print(f"Choose feature shape: {X_choose_preview.shape}")
print(f"Proof feature shape: {X_proof_preview.shape}")
print(f"Missing values in train features: {int(X_train_preview.isna().sum().sum())}")
print(f"Missing values in choose features: {int(X_choose_preview.isna().sum().sum())}")
print(f"Missing values in proof features: {int(X_proof_preview.isna().sum().sum())}")

Feature columns:
['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'log_previous_cancellations', 'log_previous_bookings_not_canceled', 'hotel_city_hotel', 'hotel_resort_hotel']
Train feature shape: (49157, 7)
Choose feature shape: (29546, 7)
Proof feature shape: (40687, 7)
Missing values in train features: 0
Missing values in choose features: 0
Missing values in proof features: 0


## 7. Decision tree: approve or disapprove each proposal

```text
Contract periods → already approved by your direct request → split completed
  ↓ review train-only observations
D1: 16,291 exact repeated train rows, no booking ID
  approve → keep every row | disapprove → pause; specify a reliable deduplication key/rule
D2: missing values are in non-candidate fields; candidate pool has none
  approve → leave missing values as-is | disapprove → pause; specify imputation/recode
D3: unusual values are non-negative; no range evidence proves errors
  approve → retain values | disapprove → pause; specify a supported correction/cap
D4: four learner-Bin-1 fields are candidate inputs, timing not independently verified
  approve → use only as provisional model candidates | disapprove → pause feature selection
D5: prior counts are highly zero-heavy
  approve → add log1p count features | disapprove → use raw counts only
D6: hotel has two training categories
  approve → one-hot encode with train categories | disapprove → pause and choose encoding
  ↓
All approved → create final feature matrices | any disapproved/pending → no final matrices, return to that branch
```

Disapproval never deletes or recodes data automatically; it records where work pauses.

### Decision 1 — retain exact repeated rows?

Observation: 16,291 training rows exactly match an earlier row across the available fields. No reservation ID is available to prove they are invalid duplicates. Proposal: retain all rows.

In [9]:
decision_duplicates = "pending"  # Change to "approve" or "disapprove".
if decision_duplicates == "approve":
    print("Approved: retain all rows; remove none.")
elif decision_duplicates == "disapprove":
    print("Paused: no rows removed. Specify a reliable booking key or deduplication rule first.")
else:
    print("Pending: no duplicate-handling choice has been applied.")

Pending: no duplicate-handling choice has been applied.


### Decision 2 — leave missing non-candidate values unchanged?

Observation: missing values are in `company`, `agent`, `country`, and 4 `children` rows in train. The provisional feature fields have no missing values in any split. Proposal: leave missing values as-is.

In [10]:
decision_missing = "pending"  # Change to "approve" or "disapprove".
if decision_missing == "approve":
    print("Approved: preserve missing values; no imputation needed for current candidate fields.")
elif decision_missing == "disapprove":
    print("Paused: no values filled. Specify which fields and the evidence-based treatment.")
else:
    print("Pending: no missing-value handling choice has been applied.")

Pending: no missing-value handling choice has been applied.


### Decision 3 — retain unusual non-negative values?

Observation: training maxima include 55 adults and ADR 5,400; the checked numeric fields have no negatives. Proposal: retain them unless a source rule confirms an error.

In [11]:
decision_ranges = "pending"  # Change to "approve" or "disapprove".
if decision_ranges == "approve":
    print("Approved: retain unusual non-negative values; no clipping applied.")
elif decision_ranges == "disapprove":
    print("Paused: no values clipped. Specify a supported validity rule first.")
else:
    print("Pending: no range-handling choice has been applied.")

Pending: no range-handling choice has been applied.


### Decision 4 — use the four Bin 1 fields as provisional candidates?

Observation: `hotel`, `is_repeated_guest`, `previous_cancellations`, and `previous_bookings_not_canceled` have no missing values. Their exact availability at the intended prediction time still needs confirmation. Proposal: allow them into model comparisons provisionally, not as production-approved inputs.

In [12]:
decision_features = "pending"  # Change to "approve" or "disapprove".
if decision_features == "approve":
    print("Approved provisionally: use the four audited fields for offline comparisons; timing caveat remains.")
elif decision_features == "disapprove":
    print("Paused: do not use these predictors until an alternative timing-safe set is agreed.")
else:
    print("Pending: no predictor set has final approval.")

Pending: no predictor set has final approval.


### Decision 5 — add log1p versions of the prior counts?

Observation: 43,348 of 49,157 training rows have zero prior cancellations; 47,813 have zero prior non-cancelled bookings. Proposal: add `log1p` versions alongside the raw counts.

In [13]:
decision_log_features = "pending"  # Change to "approve" or "disapprove".
if decision_log_features == "approve":
    print("Approved: include both log1p prior-count features in the candidate matrix.")
elif decision_log_features == "disapprove":
    print("Disapproved: use the raw prior counts only; rebuild the preview without log1p fields.")
else:
    print("Pending: log1p features remain a preview only.")

Pending: log1p features remain a preview only.


### Decision 6 — one-hot encode hotel?

Observation: train has two hotel categories, both present in choose and proof. Proposal: create two indicator columns using categories observed in train.

In [14]:
decision_hotel_encoding = "pending"  # Change to "approve" or "disapprove".
if decision_hotel_encoding == "approve":
    print("Approved: use the two train-derived hotel indicator columns.")
elif decision_hotel_encoding == "disapprove":
    print("Paused: choose an alternative encoding before creating final matrices.")
else:
    print("Pending: hotel encoding remains a preview only.")

Pending: hotel encoding remains a preview only.


### Final gate — build approved feature matrices only

Run after editing all six decision cells. If any choice is pending or disapproved, this cell will not create final feature matrices.

In [15]:
approval_choices = {
    "duplicates": decision_duplicates,
    "missing_values": decision_missing,
    "numeric_ranges": decision_ranges,
    "candidate_features": decision_features,
    "log_features": decision_log_features,
    "hotel_encoding": decision_hotel_encoding,
}

required_gates = [
    "duplicates", "missing_values", "numeric_ranges",
    "candidate_features", "hotel_encoding",
]
waiting_gates = [name for name, choice in approval_choices.items() if choice == "pending"]
disapproved_required = [name for name in required_gates if approval_choices[name] == "disapprove"]

if waiting_gates or disapproved_required:
    print("Final feature matrices not created.")
    print("Pending decisions:", ", ".join(waiting_gates) if waiting_gates else "none")
    print("Disapproved required decisions:", ", ".join(disapproved_required) if disapproved_required else "none")
elif decision_log_features not in ["approve", "disapprove"]:
    print("Final feature matrices not created. Set the log-feature decision to approve or disapprove.")
elif decision_hotel_encoding != "approve":
    print("Final feature matrices not created. Hotel encoding needs approval or an agreed alternative.")
else:
    X_train = X_train_preview.copy()
    X_choose = X_choose_preview.copy()
    X_proof = X_proof_preview.copy()
    if decision_log_features == "disapprove":
        log_columns = ["log_previous_cancellations", "log_previous_bookings_not_canceled"]
        X_train = X_train.drop(columns=log_columns)
        X_choose = X_choose.drop(columns=log_columns)
        X_proof = X_proof.drop(columns=log_columns)
    print("Required data-handling and feature choices approved.")
    print(f"Log1p features: {decision_log_features}")
    print("Final feature matrix shapes:", X_train.shape, X_choose.shape, X_proof.shape)

Final feature matrices not created.
Pending decisions: duplicates, missing_values, numeric_ranges, candidate_features, log_features, hotel_encoding
Disapproved required decisions: none


## Current result and decision status

- Data loaded and split by the requested contract periods.
- Training-period EDA and cleaning proposals are visible above. The raw data has not been changed.
- Candidate feature matrices are previews only.
- No model has been trained and no proof cancellation outcome has been inspected.

The final approval summary below will create `X_train`, `X_choose`, and `X_proof` only after every required gate is approved. If any gate is disapproved or pending, it stops and reports which decision needs discussion.